# Export a standardized QGIS map

The `openplaces.viz.qgis_map.export_qgis_map` function builds a QGIS project (`.qgz`) for a curate-stage recipe and admin unit. The generated project displays the curated output alongside the essential ingested inputs it depends on, styled consistently across all regions. This feature is opt-in and on-demand: it is **not** run automatically by `curate()`, but can be called whenever you want to generate a map for a specific run.

The framework consists of three main components demonstrated below:

- `resolve_layers`: Walks the recipe's dependency graph in pure Python to find the Parquet file paths for every layer to be displayed on the map.
- `style_registry`: Uses a CSV registry that maps a layer's canonical `(entity_type, source, role)` identity to a specific prototype layer in the QGIS template.
- `export_qgis_map`: Clones, prunes, and configures the template project to generate a tailored `.qgz` file for the given recipe and admin unit.

**Prerequisite:** A template project built via the QGIS GUI must be placed at `src/openplaces/qgis/templates/openplaces_template.qgz` to define the group structure, styling, and print layout. See `src/openplaces/qgis/templates/README.md` for template authoring guidelines. Note that `resolve_layers` and the style registry do not require this template or QGIS (they have no `qgis.core` dependency); the template is only required when calling `export_qgis_map` to generate the final project.

In [ ]:
import pandas as pd

import openplaces as op
from openplaces.viz.qgis_map import resolve_layers, style_registry

## Pick a recipe and admin unit

Any curate-stage recipe can be used. The `US_footprint-cheer-2026` recipe is the primary example and demonstrates how `combined` output datasets (where attributes and geometry are stored in a single file without a `_geo` sidecar) are handled.

In [ ]:
recipe_id = 'US_footprint-cheer-2026'
admin_id = 'US-NC-CE'

## Inspect the layers a map would include

The `resolve_layers` function returns a list of specifications for the layers to be mapped. This includes the curated output (`role='output'`), the ingest-stage inputs it depends on (`role='input'`, resolved recursively through intermediate stages), and administrative boundary context (`role='admin'`). You can pass `filter_existing=False` to inspect the full dependency graph, including layers that have not yet been generated or ingested for the selected admin unit.

In [ ]:
specs = resolve_layers(recipe_id, admin_id, filter_existing=False, verbose=True)

layers = pd.DataFrame(
    {
        'role': [s.role for s in specs],
        'entity_type': [s.entity_type for s in specs],
        'source': [s.source for s in specs],
        'depth': [s.depth for s in specs],
        'combined': [s.combined for s in specs],
        'exists': [s.exists for s in specs],
        'display_name': [s.display_name for s in specs],
    }
).sort_values(['role', 'depth'])
layers

Only layers that exist on disk for the specified admin unit are included in the generated map. By default, `export_qgis_map` calls `resolve_layers` with `filter_existing=True`.

In [ ]:
layers[layers['exists']]

## Style registry

The style registry maps each canonical `(entity_type, source, role)` combination to:
1. A prototype layer in the template
2. A layer-tree group for organization
3. The layer's default visibility (checked/unchecked)

A wildcard fallback is supported using a blank `source` field for a given `entity_type` and `role`. Support for new data sources can be added by editing the CSV registry rather than changing code.

In [ ]:
style_registry.load_registry()

In [ ]:
# Retrieve the style configuration that will be applied to the curated output layer.
output_spec = next(s for s in specs if s.role == 'output')
style_registry.get_style(output_spec.entity_type, output_spec.source, output_spec.role)

If a resolved layer does not match any row in the registry (e.g., a new `entity_type`/`source` combination), it is still included in the map using a generic fallback style. This prints a warning but does not block map generation.

## Generate the map

Generating the map requires the pre-authored QGIS template (see the prerequisite note above).

In [ ]:
try:
    qgz_path = op.export_qgis_map(recipe_id, admin_id, verbose=True)
    print(f'Wrote {qgz_path}')
except FileNotFoundError as exc:
    print(exc)
    print(
        '\nAuthor the standardized template first — see '
        'src/openplaces/qgis/templates/README.md.'
    )

When you open the generated `.qgz` project in QGIS:

- The curated output and input layers are loaded and joined (pairing attribute and geometry Parquet files, matching the pattern in `qgis/load_joined_parquet.py`).
- Layers are organized into groups and styled according to the registry.
- The map canvas extent is zoomed to the boundary of the selected admin unit.
- Custom project variables (`@recipe_id`, `@admin_id`, and `@generated_at`) are set, making them available to dynamic labels in the print layout (such as the map title).